In [22]:
%run 00_utils.ipynb

%load_ext autoreload
%autoreload 2

Root-Verzeichnis: /home/jessh/Study/Bachelor-Thesis
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
from pathlib import Path
import cv2, os
import numpy as np
from data_handler.dataset import AntiUAVDataset
import albumentations as A

from augmentations.ir_camara_effects import thermal_histogram_compression, thermal_histogram_shift, simulate_fpn_corruption, add_common_mode_noise
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider

In [24]:
train_data = AntiUAVDataset(TRAINING_DIR, debug=True)

In [25]:
def wrap(transform):
    # Falls nur ein einzelner Transform übergeben wurde, in eine Liste packen
    if not isinstance(transform, (list, tuple)):
        transform = [transform]
        
    return A.Compose(
        transform, 
        bbox_params=A.BboxParams(format='coco', label_fields=['class_labels']),
        save_applied_params=True
    )

PROB = 1.0

In [26]:
def visualise(img_org, img_aug):
    _, ax = plt.subplots(1, 3, figsize=(15, 5))
    vmin, vmax = img_org.min(), img_org.max()   # gleicher Skalenbereich für faire Vergleichbarkeit
    
    ax[0].imshow(img_org, cmap='gray', vmin=vmin, vmax=vmax); ax[0].set_title(f"original")
    ax[1].imshow(img_aug, cmap='gray', vmin=vmin, vmax=vmax); ax[1].set_title(f"augmented")
    ax[2].imshow(img_aug - img_org, cmap='seismic');          ax[2].set_title("difference")
    
    for a in ax: 
        a.axis('off')
        
    plt.tight_layout()
    plt.show()

In [27]:
img, target, img_org = train_data[100000]

In [28]:
from augmentations.ir_camara_effects import blurring_with_psf

def explore(sigma_0=2.0, rho=2.0, theta_deg=320, noise_std=0.0):
    out = blurring_with_psf(
        img_org, sigma_0=sigma_0, rho=rho,
        theta=np.radians(theta_deg), noise_std=noise_std
    )
    visualise(img_org, out)

interact(
    explore,
    sigma_0   = FloatSlider(min=0.01, max=10,  step=0.1,  value=0.0, continuous_update=False),
    rho       = FloatSlider(min=1.0, max=5,   step=0.1,  value=1.0, continuous_update=False),
    theta_deg = FloatSlider(min=0,   max=360, step=5,    value=0,   continuous_update=False),
    noise_std = FloatSlider(min=0,   max=5.0, step=0.5,  value=0.0, continuous_update=False, readout_format='.3f'),
);

interactive(children=(FloatSlider(value=0.01, continuous_update=False, description='sigma_0', max=10.0, min=0.…

In [ ]:
from augmentations.ir_camara_effects import motion_blur

def explore(tau_im=25.0, kernel_length_val=10, noise_std=0.0):
    # Wenn kernel_length_val auf 0 steht, wird None übergeben (automatische Berechnung)
    k_len = None if kernel_length_val == 0 else int(kernel_length_val)
    
    out = motion_blur(img_org, tau_im=tau_im, kernel_length=k_len, noise_std=noise_std)
    
    visualise(img_org, out)

# Interaktiver Widget Explorer (0 bei kernel_length entspricht None)
interact(
    explore,
    tau_im            = FloatSlider(min=1.0, max=60.0, step=1.0, value=25.0, continuous_update=False),
    kernel_length_val = IntSlider(  min=0,   max=50,   step=1,   value=10,   continuous_update=False, description='kernel_len'),
    noise_std         = FloatSlider(min=0.0, max=5.0,  step=0.5, value=0.0,  readout_format='.3f', continuous_update=False),
);

interactive(children=(FloatSlider(value=25.0, continuous_update=False, description='tau_im', max=60.0, min=1.0…

In [30]:
from augmentations.ir_camara_effects import fpn

def explore(sigma_gain=0.01, sigma_offset=2.0, sigma_noise=1.0, mode="pixel"):
    out = fpn(
        img_org, 
        sigma_gain=sigma_gain, 
        sigma_offset=sigma_offset, 
        sigma_noise=sigma_noise, 
        mode=mode
    )
    
    visualise(img_org, out)

# Interaktiver Widget Explorer für das neue FPN
interact(
    explore,
    sigma_gain   = FloatSlider(min=0.0, max=0.5, step=0.025,  value=0.0, readout_format='.2f', continuous_update=False),
    sigma_offset = FloatSlider(min=0.0, max=10.0, step=0.5,   value=0.0, readout_format='.1f', continuous_update=False),
    sigma_noise  = FloatSlider(min=0.0, max=15.0, step=0.5,   value=0.0, readout_format='.1f', continuous_update=False),
    mode         = widgets.Dropdown(options=["pixel", "column", "row"], value="pixel", description="mode")
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='sigma_gain', max=0.5, step=…

In [31]:
from augmentations.ir_camara_effects import thermal_histogram_shift

def explore(delta_limit=30):
    out = thermal_histogram_shift(
        img_org, 
        delta_limit
    )

    visualise(img_org, out)

# Interaktiver Widget Explorer für das neue FPN
interact(
    explore,
    delta_limit   = FloatSlider(min=0.0, max=100,   step=5,   value=0.0, readout_format='.2f', continuous_update=False),
);

interactive(children=(FloatSlider(value=0.0, continuous_update=False, description='delta_limit', step=5.0), Ou…

In [36]:
from augmentations.ir_camara_effects import salt_and_peper

def explore(p, q):
    out = salt_and_peper(
        img_org, 
        p=p, q=q, 
        s_max=255
    )

    visualise(img_org, out)


# Interaktiver Widget Explorer für das neue FPN
interact(
    explore,
    p = FloatSlider(min=0.0, max=0.025, step=0.005, value=0.0, readout_format='.3f', continuous_update=True, description='Black pixel (%)'),
    q = FloatSlider(min=0.0, max=0.025, step=0.005, value=0.0, readout_format='.3f', continuous_update=True, description='White pixel (%)'),
);

interactive(children=(FloatSlider(value=0.0, description='Black pixel (%)', max=0.025, readout_format='.3f', s…